# Airport Transfer Summary

**Grain:** pickup year/month × airport × counterpart zone × direction. The airport IDs are the NYC taxi-zone lookup entries for Newark (1), JFK (132), and LaGuardia (138). Trips with two airport endpoints are counted once as `airport_to_airport`, using the pickup airport as the airport and dropoff airport as the counterpart. `avg_total_amount` is the mean total paid, not base fare. Unknown counterpart names are retained.


In [1]:
from pathlib import Path
import sys

# Find the project even when Jupyter starts in its notebooks directory.
project_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
sys.path.insert(0, str(project_root / "notebooks" / "data_products"))
from product_utils import get_spark, register_integrated, publish_product

spark = get_spark()
register_integrated(spark)


D:\Master\data intensive\data-intensive-computing\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


## Build the product

The SQL below defines the grain and metrics. Run this cell again to refresh the Delta table.


In [2]:
# Spark SQL keeps the grouping logic visible and reusable.
query = """
WITH airport_trips AS (
    SELECT
        pickup_year,
        pickup_month,
        CASE
            WHEN pickup_location_id IN (1, 132, 138)
             AND dropoff_location_id IN (1, 132, 138) THEN 'airport_to_airport'
            WHEN pickup_location_id IN (1, 132, 138) THEN 'from_airport'
            WHEN dropoff_location_id IN (1, 132, 138) THEN 'to_airport'
        END AS direction,
        CASE WHEN pickup_location_id IN (1, 132, 138)
             THEN pickup_location_id ELSE dropoff_location_id END AS airport_location_id,
        CASE WHEN pickup_location_id IN (1, 132, 138)
             THEN pickup_zone ELSE dropoff_zone END AS airport_zone,
        CASE WHEN pickup_location_id IN (1, 132, 138)
             THEN dropoff_location_id ELSE pickup_location_id END AS counterpart_location_id,
        CASE WHEN pickup_location_id IN (1, 132, 138)
             THEN dropoff_zone ELSE pickup_zone END AS counterpart_zone,
        CASE WHEN pickup_location_id IN (1, 132, 138)
             THEN dropoff_borough ELSE pickup_borough END AS counterpart_borough,
        trip_id, trip_distance, trip_duration_minutes, total_amount
    FROM integrated_taxi_trips
    WHERE trip_id IS NOT NULL
      AND (pickup_location_id IN (1, 132, 138)
        OR dropoff_location_id IN (1, 132, 138))
)
SELECT
    pickup_year, pickup_month, direction, airport_location_id,
    COALESCE(airport_zone, 'Unknown') AS airport_zone,
    counterpart_location_id,
    COALESCE(counterpart_zone, 'Unknown') AS counterpart_zone,
    COALESCE(counterpart_borough, 'Unknown') AS counterpart_borough,
    COUNT(trip_id) AS trip_count,
    ROUND(AVG(trip_distance), 3) AS avg_trip_distance,
    ROUND(AVG(trip_duration_minutes), 3) AS avg_trip_duration_minutes,
    ROUND(AVG(total_amount), 2) AS avg_total_amount
FROM airport_trips
GROUP BY pickup_year, pickup_month, direction, airport_location_id,
         COALESCE(airport_zone, 'Unknown'), counterpart_location_id,
         COALESCE(counterpart_zone, 'Unknown'),
         COALESCE(counterpart_borough, 'Unknown')
"""
result = spark.sql(query)
product = publish_product(spark, result, "airport_transfer_summary", ())
product.printSchema()
product.show(20, truncate=False)


Saved airport_transfer_summary to D:\Master\data intensive\data-intensive-computing\data\processed\analytical_products\airport_transfer_summary
root
 |-- pickup_year: integer (nullable = true)
 |-- pickup_month: integer (nullable = true)
 |-- direction: string (nullable = true)
 |-- airport_location_id: integer (nullable = true)
 |-- airport_zone: string (nullable = true)
 |-- counterpart_location_id: integer (nullable = true)
 |-- counterpart_zone: string (nullable = true)
 |-- counterpart_borough: string (nullable = true)
 |-- trip_count: long (nullable = true)
 |-- avg_trip_distance: double (nullable = true)
 |-- avg_trip_duration_minutes: double (nullable = true)
 |-- avg_total_amount: double (nullable = true)
 |-- source_table: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- refreshed_at: timestamp (nullable = true)
 |-- schema_version: string (nullable = true)

+-----------+------------+------------+-------------------+-----------------+----------------